# BI-Ready Data Preparation (Retail Insights `data.csv`)

This notebook:
- Loads `data/raw/data.csv`
- Standardizes column names and types
- Creates **BI-consistent calculated metrics** (subtotal/discount/order_total/total)
- Flags common data quality issues
- Exports:
  - `data/clean/bi_ready_clean.csv`
  - `data/quality/data_issues.csv`
  - `data/quality/data_quality_report.json`

> Keep `data/raw/data.csv` unchanged. Work on clean outputs only.


## 0) Install dependencies (run once)

If you already installed these in your venv, you can skip this cell.

In [1]:
# If running inside Jupyter without a prepared venv, uncomment:
# !pip install pandas numpy pyarrow pandera


## 1) Configure paths

In [2]:
from pathlib import Path

RAW = Path("data/raw/data.csv")               
OUT_CLEAN  = Path("data/clean/bi_ready_clean.csv")
OUT_ISSUES = Path("data/quality/data_issues.csv")
OUT_REPORT = Path("data/quality/data_quality_report.json")

OUT_CLEAN.parent.mkdir(parents=True, exist_ok=True)
OUT_ISSUES.parent.mkdir(parents=True, exist_ok=True)

assert RAW.exists(), f"CSV not found: {RAW.resolve()}"

print("RAW:", RAW.resolve())
print("OUT_CLEAN:", OUT_CLEAN.resolve())
print("OUT_ISSUES:", OUT_ISSUES.resolve())
print("OUT_REPORT:", OUT_REPORT.resolve())


RAW: E:\Arabic Analytics Copilot\data\raw\data.csv
OUT_CLEAN: E:\Arabic Analytics Copilot\data\clean\bi_ready_clean.csv
OUT_ISSUES: E:\Arabic Analytics Copilot\data\quality\data_issues.csv
OUT_REPORT: E:\Arabic Analytics Copilot\data\quality\data_quality_report.json


## 2) Load data + quick profiling

In [12]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv(RAW)
# df2=pd.read_csv(OUT_CLEAN)
df.head()
# df2.head()


,Order No,Order Date,Customer Name,Address,City,State,Customer Type,Account Manager,Order Priority,Product Name,...,Cost Price,Retail Price,Profit Margin,Order Quantity,Sub Total,Discount %,Discount $,Order Total,Shipping Cost,Total
0,4293-1,02-09-2014,Vivek Sundaresam,"152 Bunnerong Road,Eastgardens",Sydney,NSW,Small Business,Tina Carlton,Critical,UGen Ultra Professional Cordless Optical Suite,...,$156.50,$300.97,$144.47,23.0,"$4,533.52",2%,$194.83,"$4,757.22",$7.18,"$4,291.55"
1,5001-1,24-10-2015,Shahid Hopkins,"438 Victoria Avenue,Chatswood",Sydney,NSW,Corporate,Natasha Song,Medium,Bagged Rubber Bands,...,$0.24,$1.26,$1.02,8.0,$45.20,3%,$0.00,$45.90,$0.70,$46.91
2,5004-1,13-03-2014,Dennis Pardue,"412 Brunswick St,Fitzroy",Melbourne,VIC,Consumer,Connor Betts,Not Specified,TechSavi Cordless Navigator Duo,...,$42.11,$80.98,$38.87,45.0,$873.32,4%,$72.23,$837.57,$7.18,$82.58
3,5009-1,18-02-2013,Sean Wendt,"145 Ramsay St,Haberfield",Sydney,NSW,Small Business,Phoebe Gour,Critical,Artisan Printable Repositionable Plastic Tabs,...,$5.33,$8.60,$3.27,16.0,$73.52,1%,$4.35,$740.67,$6.19,$730.92
4,5010-1,13-09-2014,Christina Vanderzanden,"188 Pitt Street,Sydney",Sydney,NSW,Small Business,Tina Carlton,Not Specified,Pizazz Drawing Pencil Set,...,$1.53,$2.78,$1.25,49.0,$138.46,7%,$5.95,$123.77,$1.34,$125.97


In [4]:
# Basic profiling snapshot
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "nulls": [int(df[c].isna().sum()) for c in df.columns],
    "null_pct": [float(df[c].isna().mean()) for c in df.columns],
    "nunique": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
profile.sort_values("null_pct", ascending=False).head(15)


,column,dtype,nulls,null_pct,nunique
3,Address,str,1,0.0002,195
17,Order Quantity,float64,1,0.0002,50
1,Order Date,str,0,0.0000,881
0,Order No,str,0,0.0000,1435
4,City,str,0,0.0000,2
5,State,str,0,0.0000,2
6,Customer Type,str,0,0.0000,4
2,Customer Name,str,0,0.0000,789
7,Account Manager,str,0,0.0000,19
8,Order Priority,str,0,0.0000,5


## 3) Helpers: parse money/percent + rename columns

In [5]:
def money_to_float(x):
    if pd.isna(x): 
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip().replace("$", "").replace(",", "")
    s = s.replace("(", "-").replace(")", "")
    try:
        return float(s)
    except ValueError:
        return np.nan

def pct_to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("%", "")
    try:
        return float(s)
    except ValueError:
        return np.nan

rename_map = {
    "Order No": "order_no",
    "Order Date": "order_date_raw",
    "Ship Date": "ship_date_raw",
    "Customer Name": "customer_name",
    "Address": "address",
    "City": "city",
    "State": "state",
    "Customer Type": "customer_type",
    "Account Manager": "account_manager",
    "Order Priority": "order_priority",
    "Product Name": "product_name",
    "Product Category": "product_category",
    "Product Container": "product_container",
    "Ship Mode": "ship_mode",
    "Order Quantity": "order_quantity_raw",
}


## 4) Clean + type conversions

In [6]:
clean = df.rename(columns=rename_map).copy()

# Parse dates (dataset format is usually DD-MM-YYYY)
clean["order_date"] = pd.to_datetime(clean["order_date_raw"], dayfirst=True, errors="coerce")
clean["ship_date"]  = pd.to_datetime(clean["ship_date_raw"],  dayfirst=True, errors="coerce")
clean["ship_delay_days"] = (clean["ship_date"] - clean["order_date"]).dt.days

# Numeric conversions (keep raw financials for audit)
clean["cost_price"]         = df["Cost Price"].map(money_to_float)
clean["retail_price"]       = df["Retail Price"].map(money_to_float)
clean["profit_margin_raw"]  = df["Profit Margin"].map(money_to_float)   # (often per-unit margin)
clean["sub_total_raw"]      = df["Sub Total"].map(money_to_float)
clean["discount_pct"]       = df["Discount %"].map(pct_to_float)
clean["discount_amount_raw"]= df["Discount $"].map(money_to_float)
clean["order_total_raw"]    = df["Order Total"].map(money_to_float)
clean["shipping_cost"]      = df["Shipping Cost"].map(money_to_float)
clean["total_raw"]          = df["Total"].map(money_to_float)

# Quantity
clean["order_quantity"] = pd.to_numeric(clean["order_quantity_raw"], errors="coerce").round().astype("Int64")

# Normalize text fields
text_cols = [
    "customer_name","address","city","state","customer_type","account_manager","order_priority",
    "product_name","product_category","product_container","ship_mode"
]
for c in text_cols:
    if c in clean.columns:
        clean[c] = clean[c].astype("string").str.strip()

# Derived date parts
clean["order_year"] = clean["order_date"].dt.year
clean["order_month"] = clean["order_date"].dt.month
clean["order_quarter"] = clean["order_date"].dt.quarter

clean.head()


,order_no,order_date_raw,customer_name,address,city,state,customer_type,account_manager,order_priority,product_name,...,sub_total_raw,discount_pct,discount_amount_raw,order_total_raw,shipping_cost,total_raw,order_quantity,order_year,order_month,order_quarter
0,4293-1,02-09-2014,Vivek Sundaresam,"152 Bunnerong Road,Eastgardens",Sydney,NSW,Small Business,Tina Carlton,Critical,UGen Ultra Professional Cordless Optical Suite,...,4533.52,2.0,194.83,4757.22,7.18,4291.55,23,2014,9,3
1,5001-1,24-10-2015,Shahid Hopkins,"438 Victoria Avenue,Chatswood",Sydney,NSW,Corporate,Natasha Song,Medium,Bagged Rubber Bands,...,45.20,3.0,0.00,45.90,0.70,46.91,8,2015,10,4
2,5004-1,13-03-2014,Dennis Pardue,"412 Brunswick St,Fitzroy",Melbourne,VIC,Consumer,Connor Betts,Not Specified,TechSavi Cordless Navigator Duo,...,873.32,4.0,72.23,837.57,7.18,82.58,45,2014,3,1
3,5009-1,18-02-2013,Sean Wendt,"145 Ramsay St,Haberfield",Sydney,NSW,Small Business,Phoebe Gour,Critical,Artisan Printable Repositionable Plastic Tabs,...,73.52,1.0,4.35,740.67,6.19,730.92,16,2013,2,1
4,5010-1,13-09-2014,Christina Vanderzanden,"188 Pitt Street,Sydney",Sydney,NSW,Small Business,Tina Carlton,Not Specified,Pizazz Drawing Pencil Set,...,138.46,7.0,5.95,123.77,1.34,125.97,49,2014,9,3


## 5) BI-consistent calculated metrics (official truth)

In [7]:
# Calculated (consistent) metrics
clean["sub_total"]       = clean["retail_price"] * clean["order_quantity"].astype(float)
clean["discount_amount"] = clean["sub_total"] * (clean["discount_pct"] / 100.0)
clean["order_total"]     = clean["sub_total"] - clean["discount_amount"]
clean["total"]           = clean["order_total"] + clean["shipping_cost"]

# Cost & profit estimates (adjust later if your business defines shipping differently)
clean["cogs"] = clean["cost_price"] * clean["order_quantity"].astype(float)
clean["gross_profit"] = clean["order_total"] - clean["cogs"]
clean["profit_after_shipping"] = clean["gross_profit"] - clean["shipping_cost"]

# A computed unit margin check
clean["profit_margin_calc"] = clean["retail_price"] - clean["cost_price"]

clean[[
    "sub_total_raw","sub_total",
    "discount_amount_raw","discount_amount",
    "order_total_raw","order_total",
    "total_raw","total"
]].head(10)


,sub_total_raw,sub_total,discount_amount_raw,discount_amount,order_total_raw,order_total,total_raw,total
0,4533.52,6922.31,194.83,138.4462,4757.22,6783.8638,4291.55,6791.0438
1,45.20,10.08,0.00,0.3024,45.90,9.7776,46.91,10.4776
2,873.32,3644.10,72.23,145.7640,837.57,3498.3360,82.58,3505.5160
3,73.52,137.60,4.35,1.3760,740.67,136.2240,730.92,142.4140
4,138.46,136.22,5.95,9.5354,123.77,126.6846,125.97,128.0246
5,197.36,327.60,12.98,26.2080,183.58,301.3920,189.43,312.5420
6,65.50,117.90,5.90,4.7160,59.62,113.1840,60.45,113.9840
7,24.92,89.08,2.49,1.7816,22.43,87.2984,27.43,88.0984
8,194.36,155.75,15.93,15.5750,178.12,140.1750,189.45,147.1450
9,1079.62,12640.74,30.83,505.6296,1753.33,12135.1104,1705.42,12142.2904


## 6) Data quality rules + issues table

In [8]:
def abs_close(a, b, abs_tol=0.02, rel_tol=0.02):
    # tolerance for rounding / currency issues
    return (np.abs(a-b) <= abs_tol) | (np.abs(a-b) <= rel_tol*np.maximum(1.0, np.abs(b)))

clean["q_ship_before_order"] = clean["ship_delay_days"] < 0
clean["q_discount_pct_bad"] = ~clean["discount_pct"].between(0, 100, inclusive="both")

# Reconciliation checks vs raw numbers
clean["q_sub_total_mismatch"] = ~abs_close(clean["sub_total_raw"], clean["sub_total"])
clean["q_discount_amount_mismatch"] = ~abs_close(clean["discount_amount_raw"], clean["discount_amount"], abs_tol=0.05, rel_tol=0.03)
clean["q_order_total_mismatch"] = ~abs_close(clean["order_total_raw"], clean["order_total"])
clean["q_total_mismatch"] = ~abs_close(clean["total_raw"], clean["total"])

# Profit margin mismatch (if dataset margin is intended to be retail-cost)
clean["q_profit_margin_mismatch"] = ~abs_close(clean["profit_margin_raw"], clean["profit_margin_calc"], abs_tol=0.02, rel_tol=0.01)

flag_cols = [c for c in clean.columns if c.startswith("q_")]
clean["q_any"] = clean[flag_cols].any(axis=1)

issues = clean.loc[clean["q_any"], ["order_no","order_date","ship_date","customer_name","product_name"] + flag_cols].copy()

print("Rows:", len(clean))
print("Issue rows:", len(issues), f"({len(issues)/max(1,len(clean)):.2%})")
issues.head(20)


Rows: 5000
Issue rows: 4998 (99.96%)


,order_no,order_date,ship_date,customer_name,product_name,q_ship_before_order,q_discount_pct_bad,q_sub_total_mismatch,q_discount_amount_mismatch,q_order_total_mismatch,q_total_mismatch,q_profit_margin_mismatch
0,4293-1,2014-09-02,2014-09-04,Vivek Sundaresam,UGen Ultra Professional Cordless Optical Suite,False,False,True,True,True,True,False
1,5001-1,2015-10-24,2015-10-26,Shahid Hopkins,Bagged Rubber Bands,False,False,True,True,True,True,False
2,5004-1,2014-03-13,2014-03-13,Dennis Pardue,TechSavi Cordless Navigator Duo,False,False,True,True,True,True,False
3,5009-1,2013-02-18,2013-02-20,Sean Wendt,Artisan Printable Repositionable Plastic Tabs,False,False,True,True,True,True,False
4,5010-1,2014-09-13,2014-09-17,Christina Vanderzanden,Pizazz Drawing Pencil Set,False,False,False,True,True,False,False
5,5011-1,2013-11-24,2013-11-26,Patrick OBrill,"Alto Parchment Paper, Assorted Colors",False,False,True,True,True,True,False
6,5012-1,2014-09-07,2014-09-07,Eugene Moren,Smiths Metal Binder Clips,False,False,True,True,True,True,False
7,5012-1,2015-01-28,2015-01-29,Phillip Flathmann,Smiths Metal Binder Clips,False,False,True,True,True,True,False
8,5014-1,2014-06-08,2014-06-10,John Lee,Artisan Hole Reinforcements,False,False,True,False,True,True,False
9,5015-1,2014-10-02,2014-10-04,Ritsa Hightower,UGen Ultra Professional Cordless Optical Suite,False,False,True,True,True,True,False


## 7) Export BI-ready dataset + reports

In [9]:
# Keep a clean BI table + audit columns (raw financials) + flags
keep_cols = [
    "order_no","order_date","ship_date","ship_delay_days",
    "customer_name","address","city","state","customer_type","account_manager","order_priority",
    "product_name","product_category","product_container","ship_mode",
    "order_quantity","cost_price","retail_price",
    "discount_pct","shipping_cost",
    # BI metrics
    "sub_total","discount_amount","order_total","total",
    "cogs","gross_profit","profit_after_shipping",
    # raw for audit
    "sub_total_raw","discount_amount_raw","order_total_raw","total_raw",
    # date parts
    "order_year","order_month","order_quarter",
    # flags
    *flag_cols
]

bi = clean[keep_cols].copy()

report = {
    "rows": int(len(bi)),
    "issues_rows": int(len(issues)),
    "issues_rate": float(len(issues) / max(1, len(bi))),
    "missing_address": int(bi["address"].isna().sum()),
    "missing_order_quantity": int(bi["order_quantity"].isna().sum()),
    "ship_before_order": int(bi["q_ship_before_order"].sum()),
    "sub_total_mismatch": int(bi["q_sub_total_mismatch"].sum()),
    "discount_amount_mismatch": int(bi["q_discount_amount_mismatch"].sum()),
    "order_total_mismatch": int(bi["q_order_total_mismatch"].sum()),
    "total_mismatch": int(bi["q_total_mismatch"].sum()),
    "profit_margin_mismatch": int(bi["q_profit_margin_mismatch"].sum()),
    "distinct_orders": int(bi["order_no"].nunique()),
    "avg_lines_per_order": float(len(bi) / bi["order_no"].nunique()),
}

bi.to_csv(OUT_CLEAN, index=False)
issues.to_csv(OUT_ISSUES, index=False)
OUT_REPORT.write_text(json.dumps(report, indent=2), encoding="utf-8")

report


{'rows': 5000,
 'issues_rows': 4998,
 'issues_rate': 0.9996,
 'missing_address': 1,
 'missing_order_quantity': 1,
 'ship_before_order': 75,
 'sub_total_mismatch': 4930,
 'discount_amount_mismatch': 4897,
 'order_total_mismatch': 4931,
 'total_mismatch': 4907,
 'profit_margin_mismatch': 20,
 'distinct_orders': 1435,
 'avg_lines_per_order': 3.484320557491289}